# Prompt to Picture — Text-to-Image Generator in Google Colab

Turn a written prompt into a picture using a small, fast Stable Diffusion
model (`SD-Turbo`) running fully locally — no API key, no account, same
"runs right here" pattern as the Company Policy Assistant.

**⚠️ This needs a GPU runtime.** Before running anything: **Runtime →
Change runtime type → T4 GPU** (still free), then come back and run the
cells below in order, top to bottom.

You'll need:
- A GPU runtime (see above)
- Nothing else — no API key, no sign-up


## 1. Install dependencies

In [ ]:
!pip install diffusers transformers accelerate torch streamlit --quiet


## 2. Check you have a GPU

If this prints "No GPU detected," stop here and go to **Runtime → Change
runtime type → T4 GPU**, then **Runtime → Restart session** and re-run
from Cell 1. Image generation on CPU is painfully slow.


In [ ]:
import torch

if torch.cuda.is_available():
    print("GPU detected:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected! Go to Runtime -> Change runtime type -> T4 GPU, then Runtime -> Restart session.")


## 3. Write `app.py`

The Streamlit app itself: a prompt box, a Generate button, and the image
model. Everything runs locally — the model (`stabilityai/sd-turbo`)
downloads automatically the first time it runs.


In [ ]:
%%writefile app.py
"""
Prompt to Picture — Streamlit app, fully local text-to-image
================================================================
Turns a written description into a picture using a small, fast Stable
Diffusion model (SD-Turbo) running right here on the Colab GPU. Same
"runs fully locally, no API key" pattern as the Company Policy Assistant
— just swapping a text-answering model for an image-generating one.

How it works, in one sentence: the model starts from random noise and,
guided by your prompt, removes a bit of that noise each step until a
picture emerges. SD-Turbo is a *distilled* model built to do this in
just 1-4 steps instead of the 20-50 a normal Stable Diffusion model
needs — that's what makes it fast enough for a live classroom demo.

SETUP:
    pip install diffusers transformers accelerate torch streamlit --break-system-packages

IMPORTANT — THIS NEEDS A GPU. Before running anything in Colab:
    Runtime -> Change runtime type -> T4 GPU (still free)
Without a GPU this will be extremely slow (minutes per image) or may
run out of memory entirely.

RUN THIS WITH:
    streamlit run app.py

(Do NOT run it with `python app.py` — Streamlit apps must be launched
with the `streamlit run` command.)
"""

from io import BytesIO

import streamlit as st
import torch
from diffusers import AutoPipelineForText2Image

# ============================================================
# CONFIGURATION
# ============================================================

# A small, fast, open Stable Diffusion model — distilled to generate a
# good image in just 1-4 steps instead of the usual 20-50. Free for
# research/non-commercial use (Stability AI's community license).
MODEL_ID = "stabilityai/sd-turbo"


# ============================================================
# 1. LOAD THE IMAGE MODEL (once per session, cached)
# ============================================================
# @st.cache_resource = "run this once, keep the result in memory, hand
# back the same object on every rerun" — without it, Streamlit would
# reload the multi-gigabyte model on every single click, since it
# reruns the whole script top-to-bottom each time.

@st.cache_resource(show_spinner="Downloading and loading the image model (first run only, a few minutes)...")
def load_pipeline():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.float16 if device == "cuda" else torch.float32
    pipe = AutoPipelineForText2Image.from_pretrained(MODEL_ID, torch_dtype=dtype)
    pipe = pipe.to(device)
    return pipe, device


# ============================================================
# 2. PAGE SETUP
# ============================================================

st.set_page_config(page_title="Image Generator", page_icon="\U0001F3A8")
st.title("Prompt to Picture")
st.caption("Type a description below — a locally-running Stable Diffusion model turns it into an image.")

pipe, device = load_pipeline()

if device == "cpu":
    st.warning(
        "No GPU detected — image generation will be extremely slow or may fail. "
        "Runtime → Change runtime type → T4 GPU, then Runtime → Restart session and re-run this app."
    )
else:
    st.caption(f"Running on GPU ({torch.cuda.get_device_name(0)}) · model: {MODEL_ID}")

if "gallery" not in st.session_state:
    st.session_state.gallery = []  # list of (prompt, PIL image) for this session


# ============================================================
# 3. THE PROMPT FORM — everything the student can control
# ============================================================

with st.form("prompt_form"):
    prompt = st.text_area(
        "Describe the image you want",
        placeholder="A cozy South Indian filter coffee stall at dawn, steam rising from a tumbler, warm golden light, cinematic photography",
    )
    col1, col2 = st.columns(2)
    with col1:
        steps = st.slider("Steps (more = slower, usually better)", 1, 4, 2)
    with col2:
        seed = st.number_input("Seed (same seed + same prompt = same image)", value=0, step=1)
    submitted = st.form_submit_button("Generate")


# ============================================================
# 4. GENERATE — the actual prompt -> image call
# ============================================================

if submitted and prompt.strip():
    generator = torch.Generator(device=device).manual_seed(int(seed))
    with st.spinner("Generating..."):
        image = pipe(
            prompt=prompt,
            num_inference_steps=steps,
            guidance_scale=0.0,  # SD-Turbo is trained for guidance-free generation
            generator=generator,
        ).images[0]
    st.session_state.gallery.insert(0, (prompt, image))


# ============================================================
# 5. SHOW THE RESULT + THIS SESSION'S GALLERY
# ============================================================

if st.session_state.gallery:
    latest_prompt, latest_image = st.session_state.gallery[0]
    st.image(latest_image, caption=latest_prompt, use_container_width=True)

    buf = BytesIO()
    latest_image.save(buf, format="PNG")
    st.download_button("Download this image", data=buf.getvalue(), file_name="generated_image.png", mime="image/png")

    if len(st.session_state.gallery) > 1:
        st.subheader("Earlier in this session")
        cols = st.columns(4)
        for i, (p, img) in enumerate(st.session_state.gallery[1:9]):
            with cols[i % 4]:
                label = p[:40] + ("..." if len(p) > 40 else "")
                st.image(img, caption=label, use_container_width=True)
else:
    st.info("Type a prompt above and click Generate to create your first image.")


## 4. Launch Streamlit in the background

The first click of "Generate" will be slower — that's the model
downloading (a few GB, one time) and loading onto the GPU. After that it
stays loaded for the rest of the session.


In [ ]:
!streamlit run app.py &>/content/logs.txt &


## 5. Download cloudflared

Colab has no public address of its own. Cloudflare's free quick tunnel
hands you one — no password step, and far more reliable than
`localtunnel` for apps that lazy-load JavaScript (like Streamlit).


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared


## 6. Launch the tunnel and get your link

Starts the tunnel in the background, waits a few seconds, then prints
your public URL — a `https://....trycloudflare.com` link. Click it to
open the app directly.

If this cell prints nothing under "Your app is live at," just re-run it
— the tunnel can occasionally take a couple of extra seconds to connect.


In [ ]:
import re, subprocess, time

subprocess.Popen(
    "./cloudflared tunnel --url http://localhost:8501 > cloudflared_logs.txt 2>&1",
    shell=True,
)
time.sleep(8)

with open("cloudflared_logs.txt") as f:
    logs = f.read()

match = re.search(r"https://[a-zA-Z0-9.-]*\.trycloudflare\.com", logs)
if match:
    print("Your app is live at:", match.group(0))
else:
    print("Tunnel still connecting -- re-run this cell in a few seconds.")


## Troubleshooting

- **"No GPU detected"** — Runtime → Change runtime type → T4 GPU, then
  Runtime → Restart session, then re-run from Cell 1. This app is not
  usable on a plain CPU runtime.
- **Generation is very slow or the session runs out of memory** — lower
  the "Steps" slider to 1 or 2 in the app, or Runtime → Restart session
  to clear GPU memory and try again.
- **The image comes out solid black** — Stable Diffusion's built-in
  safety filter occasionally (and incorrectly) flags an innocent prompt.
  Try rephrasing the prompt slightly and generate again.
- **Tunnel link doesn't load** — Cell 4 (Streamlit) may still be
  starting; wait ~10 seconds and retry Cell 6, or check
  `/content/logs.txt` for errors: `!cat /content/logs.txt`
- **New tunnel URL each time** — expected; re-run Cell 6 whenever you
  restart Streamlit and share the newest link.
